In [0]:
--Tabla 1: silver_matchs-------------------------------------------------------------------
WITH parsed_matches AS (
    SELECT 
        *,
        FROM_JSON(match, 'array<struct<`@context`:string, `@type`:string, name:string, description:string, sport:string, eventStatus:string, startDate:string, endDate:string, url:string, attendee:array<struct<`@type`:string, name:string, jobTitle:string>>, homeTeam:struct<`@type`:string, name:string, image:string>, awayTeam:struct<`@type`:string, name:string, image:string>, location:struct<`@type`:string, name:string>, organizer:struct<`@type`:string, name:string>, subEvent:array<struct<`@type`:string, name:string, startDate:string, location:struct<`@type`:string>, attendee:struct<`@type`:string, name:string>>>>>') AS match_json
    FROM futbol.bronze_matchs
),
cleaned_data AS (
    SELECT
        *,
        REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(name, 'Ã³', 'o'), 'Ã©', 'e'), 'Ã¡', 'a'), 'Ã­', 'i'), 'Ãº', 'u'), 'Ã±', 'n'), 'Â°', '°'), 'Ã\\.', 'Á.'), 'Ã ', 'Á ') AS clean_name,
        REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(league, 'Ã³', 'o'), 'Ã©', 'e'), 'Ã¡', 'a'), 'Ã­', 'i'), 'Ãº', 'u'), 'Ã±', 'n'), 'Â°', '°'), 'Ã\\.', 'Á.'), 'Ã ', 'Á ') AS clean_league,
        REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(match_json[0].awayTeam.name, 'Ã³', 'o'), 'Ã©', 'e'), 'Ã¡', 'a'), 'Ã­', 'i'), 'Ãº', 'u'), 'Ã±', 'n'), 'Â°', '°'), 'Ã\\.', 'Á.'), 'Ã ', 'Á ') AS clean_away_team,
        REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(match_json[0].homeTeam.name, 'Ã³', 'o'), 'Ã©', 'e'), 'Ã¡', 'a'), 'Ã­', 'i'), 'Ãº', 'u'), 'Ã±', 'n'), 'Â°', '°'), 'Ã\\.', 'Á.'), 'Ã ', 'Á ') AS clean_home_team,
        REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(match_json[0].location.name, 'Ã³', 'o'), 'Ã©', 'e'), 'Ã¡', 'a'), 'Ã­', 'i'), 'Ãº', 'u'), 'Ã±', 'n'), 'Â°', '°'), 'Ã\\.', 'Á.'), 'Ã ', 'Á ') AS clean_stadium
    FROM parsed_matches
)
SELECT DISTINCT
    REPLACE(substring_index(match_json[0].url, '-', -1), "/", "") AS match_id,
    clean_name AS name,
    REGEXP_REPLACE(clean_league, 'Ã¢', 'á') AS league,
    REGEXP_REPLACE(clean_away_team, 'Ã¢', 'á') AS away_team,
    REGEXP_REPLACE(clean_home_team, 'Ã¢', 'á') AS home_team,
    REGEXP_REPLACE(clean_stadium, 'Ã¢', 'á') AS stadium,
    CAST(match_json[0].startDate AS TIMESTAMP) AS start_date,
    CAST(match_json[0].endDate AS TIMESTAMP) AS end_date,
    REGEXP_REPLACE(match_json[0].url, '[^a-zA-Z0-9:/._-]', '') AS url
FROM cleaned_data
ORDER BY start_date DESC;


--Tabla 2: silver_match_events-------------------------------------------------------------------
WITH parsed_matches AS (
    SELECT 
        *,
        FROM_JSON(match, 'array<struct<`@context`:string, `@type`:string, name:string, description:string, sport:string, eventStatus:string, startDate:string, endDate:string, url:string, attendee:array<struct<`@type`:string, name:string, jobTitle:string>>, homeTeam:struct<`@type`:string, name:string, image:string>, awayTeam:struct<`@type`:string, name:string, image:string>, location:struct<`@type`:string, name:string>, organizer:struct<`@type`:string, name:string>, subEvent:array<struct<`@type`:string, name:string, startDate:string, location:struct<`@type`:string>, attendee:struct<`@type`:string, name:string>>>>>') AS match_json
    FROM futbol.bronze_matchs
),
exploded_events AS (
    SELECT
        *,
        sub_event
    FROM parsed_matches
    LATERAL VIEW EXPLODE(match_json[0].subEvent) exploded AS sub_event
),
cleaned_events AS (
    SELECT
        *,
        REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(sub_event.attendee.name, 'Ã³', 'o'), 'Ã©', 'e'), 'Ã¡', 'a'), 'Ã­', 'i'), 'Ãº', 'u'), 'Ã±', 'n'), 'Â°', '°'), 'Ã\\.', 'Á.'), 'Ã ', 'Á ') AS clean_event_team,
        REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(sub_event.name, 'Ã³', 'o'), 'Ã©', 'e'), 'Ã¡', 'a'), 'Ã­', 'i'), 'Ãº', 'u'), 'Ã±', 'n'), 'Â°', '°'), 'Ã\\.', 'Á.'), 'Ã ', 'Á ') AS clean_description
    FROM exploded_events
)
SELECT DISTINCT
    REPLACE(substring_index(match_json[0].url, '-', -1), "/", "") AS match_id,
    CAST(sub_event.startDate AS TIMESTAMP) AS event_time,
    clean_event_team AS event_team,
    clean_description AS description,
    CASE
        WHEN sub_event.name IS NULL THEN ""
        WHEN sub_event.name LIKE 'Gol%' THEN 'GOL'
        WHEN sub_event.name LIKE 'Amonest%' THEN 'TARJETA AMARILLA'
        WHEN sub_event.name LIKE 'Expulsi%' THEN 'TARJETA ROJA'
        WHEN sub_event.name LIKE 'Sale%' THEN 'CAMBIO'
        ELSE 'OTHER'
    END AS event_type
FROM cleaned_events
ORDER BY event_time DESC;


--Tabla 3: silver_match_players-------------------------------------------------------------------
WITH parsed_matches AS (
    SELECT 
        *,
        FROM_JSON(match, 'array<struct<`@context`:string, `@type`:string, name:string, description:string, sport:string, eventStatus:string, startDate:string, endDate:string, url:string, attendee:array<struct<`@type`:string, name:string, jobTitle:string>>, homeTeam:struct<`@type`:string, name:string, image:string>, awayTeam:struct<`@type`:string, name:string, image:string>, location:struct<`@type`:string, name:string>, organizer:struct<`@type`:string, name:string>, subEvent:array<struct<`@type`:string, name:string, startDate:string, location:struct<`@type`:string>, attendee:struct<`@type`:string, name:string>>>, `@graph`:array<struct<`@type`:string, name:string, athlete:array<struct<`@type`:string, name:string, roleName:string>>>>>>') AS match_json
    FROM futbol.bronze_matchs
),
exploded_players AS (
    SELECT
        *,
        team,
        team_players
    FROM parsed_matches
    LATERAL VIEW EXPLODE(match_json[1].`@graph`) t AS team
    LATERAL VIEW EXPLODE(team.athlete) p AS team_players
    WHERE team.`@type` = 'SportsTeam'
),
cleaned_players AS (
    SELECT
        *,
        REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(team.name, 'Ã³', 'o'), 'Ã©', 'e'), 'Ã¡', 'a'), 'Ã­', 'i'), 'Ãº', 'u'), 'Ã±', 'n'), 'Â°', '°'), 'Ã\\.', 'Á.'), 'Ã ', 'Á ') AS clean_team_name,
        REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(team_players.name, 'Ã³', 'o'), 'Ã©', 'e'), 'Ã¡', 'a'), 'Ã­', 'i'), 'Ãº', 'u'), 'Ã±', 'n'), 'Â°', '°'), 'Ã\\.', 'Á.'), 'Ã ', 'Á ') AS clean_player_name,
        REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(team_players.roleName, 'Ã³', 'o'), 'Ã©', 'e'), 'Ã¡', 'a'), 'Ã­', 'i'), 'Ãº', 'u'), 'Ã±', 'n'), 'Â°', '°'), 'Ã\\.', 'Á.'), 'Ã ', 'Á ') AS clean_player_position
    FROM exploded_players
)
SELECT DISTINCT
    REPLACE(substring_index(match_json[0].url, '-', -1), '/', '') AS match_id,
    clean_team_name AS team_name,
    clean_player_name AS player_name,
    clean_player_position AS player_position
FROM cleaned_players
ORDER BY match_id DESC;


--Tabla 4: silver_match_referees-------------------------------------------------------------------
WITH parsed_matches AS (
    SELECT 
        *,
        FROM_JSON(match, 'array<struct<`@context`:string, `@type`:string, name:string, description:string, sport:string, eventStatus:string, startDate:string, endDate:string, url:string, attendee:array<struct<`@type`:string, name:string, jobTitle:string>>, homeTeam:struct<`@type`:string, name:string, image:string>, awayTeam:struct<`@type`:string, name:string, image:string>, location:struct<`@type`:string, name:string>, organizer:struct<`@type`:string, name:string>, subEvent:array<struct<`@type`:string, name:string, startDate:string, location:struct<`@type`:string>, attendee:struct<`@type`:string, name:string>>>>>') AS match_json
    FROM futbol.bronze_matchs
),
exploded_referees AS (
    SELECT
        *,
        referee
    FROM parsed_matches
    LATERAL VIEW EXPLODE(match_json[0].attendee) t AS referee
),
cleaned_referees AS (
    SELECT
        *,
        REGEXP_REPLACE(
            REGEXP_REPLACE(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(
                        REGEXP_REPLACE(
                            REGEXP_REPLACE(
                                REGEXP_REPLACE(
                                    REGEXP_REPLACE(
                                        REGEXP_REPLACE(referee.name, 'Ã³', 'o'), 
                                    'Ã©', 'e'), 
                                'Ã¡', 'a'), 
                            'Ã­', 'i'), 
                        'Ãº', 'u'), 
                    'Ã±', 'n'), 
                'Â°', '°'), 
            'Ã\\.', 'Á.'), 
        'Ã ', 'Á ') AS clean_name,
        REGEXP_REPLACE(
            REGEXP_REPLACE(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(
                        REGEXP_REPLACE(
                            REGEXP_REPLACE(
                                REGEXP_REPLACE(
                                    REGEXP_REPLACE(
                                        REGEXP_REPLACE(referee.jobTitle, 'Ã³', 'o'), 
                                    'Ã©', 'e'), 
                                'Ã¡', 'a'), 
                            'Ã­', 'i'), 
                        'Ãº', 'u'), 
                    'Ã±', 'n'), 
                'Â°', '°'), 
            'Ã\\.', 'Á.'), 
        'Ã ', 'Á ') AS clean_role
    FROM exploded_referees
)
SELECT DISTINCT
    REPLACE(substring_index(match_json[0].url, '-', -1), '/', '') AS match_id,
    REGEXP_REPLACE(clean_name, 'Ã¢', 'á') AS name,
    REGEXP_REPLACE(clean_role, 'Ã¢', 'á') AS role
FROM cleaned_referees
ORDER BY match_id DESC;


SELECT
    DISTINCT league
FROM
    workspace.futbol.silver_matchs
;

SELECT * FROM workspace.futbol.silver_matchs LIMIT 5;
SELECT * FROM workspace.futbol.silver_match_events LIMIT 5;
SELECT * FROM workspace.futbol.silver_players LIMIT 5;